In [ ]:
%pip install -U langchain langchain-core langchain-upstage langchain-pinecone pinecone python-dotenv

In [ ]:
%pip install -U langchain-community


In [ ]:
import os
from typing import List, Tuple, Any

from dotenv import load_dotenv
from pydantic import ConfigDict
from pinecone import Pinecone

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_pinecone import PineconeVectorStore
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


# -----------------------------
# 1) 환경변수 로드
# -----------------------------
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROUNDLINE_INDEX = os.getenv("GROUNDLINE_INDEX")
BROADCOM_INDEX = os.getenv("BROADCOM_INDEX")
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY가 없습니다.")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY가 없습니다.")
if not GROUNDLINE_INDEX or not BROADCOM_INDEX:
    raise ValueError("GROUNDLINE_INDEX / BROADCOM_INDEX가 필요합니다.")


# -----------------------------
# 2) 임베딩 / LLM
# -----------------------------
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
llm = ChatUpstage()


# -----------------------------
# 3) Pinecone 인덱스 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)

groundline_index = pc.Index(GROUNDLINE_INDEX)
broadcom_index = pc.Index(BROADCOM_INDEX)

groundline_db = PineconeVectorStore(
    index=groundline_index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)

broadcom_db = PineconeVectorStore(
    index=broadcom_index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)


# -----------------------------
# 4) 멀티 인덱스 Retriever
# -----------------------------
class MultiPineconeRetriever(BaseRetriever):
    """
    두 개 이상의 PineconeVectorStore를 동시에 검색한 뒤,
    score 기준으로 정렬해서 상위 문서를 반환하는 커스텀 Retriever
    """

    vectorstores: List[Any]
    k_each: int = 5       # 각 인덱스에서 가져올 개수
    final_k: int = 5      # 최종 반환 개수

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        all_docs_with_scores: List[Tuple[Document, float]] = []

        for vs in self.vectorstores:
            # PineconeVectorStore의 similarity_search_with_score 사용
            # score는 vector store 구현에 따라 거리/유사도 의미가 다를 수 있으므로
            # 실제 결과를 보고 정렬 방향을 확인하는 것이 안전함
            docs_with_scores = vs.similarity_search_with_score(query, k=self.k_each)
            all_docs_with_scores.extend(docs_with_scores)

        # 점수 정렬
        # 현재 사용 코드 흐름에 맞춰 reverse=True 유지
        # 만약 결과가 이상하면 reverse=False로 바꿔 테스트하세요.
        all_docs_with_scores = sorted(
            all_docs_with_scores,
            key=lambda x: x[1],
            reverse=True,
        )

        # 최종 상위 문서 추출
        top_docs = []
        for rank, (doc, score) in enumerate(all_docs_with_scores[: self.final_k], start=1):
            # 출처 확인용 metadata 추가
            doc.metadata["retrieval_score"] = score
            doc.metadata["retrieval_rank"] = rank
            top_docs.append(doc)
            print("doc\n\n",doc.page_content)

        return top_docs


multi_retriever = MultiPineconeRetriever(
    vectorstores=[groundline_db, broadcom_db],
    k_each=5,
    final_k=5,
)


# -----------------------------
# 5) 검색된 Document 리스트를 문자열 context로 변환
# -----------------------------
# def format_docs(docs: List[Document]) -> str:
#     formatted = []
#     for i, doc in enumerate(docs, start=1):
#         source = doc.metadata.get("source", "출처 미상")
#         score = doc.metadata.get("retrieval_score", "")
#         formatted.append(
#             f"[문서 {i}] (source: {source}, score: {score})\n{doc.page_content}"
#         )
#     return "\n\n".join(formatted)


def format_docs(docs: List[Document]) -> str:

    formatted_docs = []

    for doc in docs:

        regulation = doc.metadata.get("regulation_name", "")
        article = doc.metadata.get("article", "")
        # clause = doc.metadata.get("clause", "")
        # source = doc.metadata.get("source", "")

        text = f"""
규정명 : {regulation}
조문: {article} 
본문:
{doc.page_content}
"""

        formatted_docs.append(text)

    return "\n\n".join(formatted_docs)


# def extract_metadata(docs):

#     doc = docs[0]   # 가장 관련 높은 문서

#     return {
#         "context": "\n\n".join([d.page_content for d in docs]),
#         "regulation": doc.metadata.get("regulation_name", ""),
#         "article": doc.metadata.get("article", ""),
#         # "clause": doc.metadata.get("clause", "")
#     }    


def extract_metadata(data):
    docs = data["docs"]
    question = data["input"]

    if not docs:
        return {
            "context": "",
            "regulation": "관련 규정 또는 기술기준 확인 불가",
            "article": "",
            "input": question,
        }

    top_doc = docs[0]

    return {
        "context": format_docs(docs),
        "regulation": top_doc.metadata.get("regulation_name", "관련 규정 또는 기술기준 확인 불가"),
        "article": top_doc.metadata.get("article", ""),
        "input": question,
    }

# -----------------------------
# 5) QA 프롬프트
# -----------------------------
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 통신설비 기술 질의응답 전문가다.
반드시 제공된 context만 근거로 답변하라.
context에 없으면 추측하지 말고 "관련 규정 또는 기술기준 확인 불가, 제공된 문서에서 확인되지 않습니다."라고 답하라.
답변은 한국어로, 핵심부터 간결하게 작성하라.

반드시 아래 형식으로만 답하라.

{regulation} {article}, 답변내용

예시:
접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준 제8조, 통신설비의 접지저항은 10Ω 이하로 해야 합니다.

<context>
{context}
</context>
""".strip(),
        ),
        ("human", "{input}"),
    ]
)

# qa_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# 너는 통신설비 기술 질의응답 전문가다.
# 반드시 제공된 context만 근거로 답변하라.
# context에 없으면 추측하지 말고 "제공된 문서에서 확인되지 않습니다."라고 답하라.
# 답변은 한국어로, 핵심부터 간결하게 작성하라.
# 답변 형식은 반드시 아래 형식을 사용하라.

# <답변 형식>
# 관련 규정 또는 기술기준 제○조 ○항, 응답내용
# </답변 형식>

# <context>
# {context}
# </context>
# """.strip(),
#         ),
#         ("human", "{input}"),
#     ]
# )

# qa_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# 너는 통신설비 기술 질의응답 전문가다.
# 반드시 제공된 context만 근거로 답변하라.
# context에 없으면 추측하지 말고 "제공된 문서에서 확인되지 않습니다."라고 답하라.
# 답변은 한국어로, 핵심부터 간결하게 작성하라.

# 답변 형식:
# {regulation} {article}, 응답내용

# <context>
# {context}
# </context>
# """.strip(),
#         ),
#         ("human", "{input}"),
#     ]
# )


# # -----------------------------
# # 6) 문서 결합 체인 + Retrieval 체인
# # -----------------------------
# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
# rag_chain = create_retrieval_chain(multi_retriever, question_answer_chain)

# prompt = ChatPromptTemplate.from_template("""
# 너는 통신설비 기술 질의응답 전문가다.
# 반드시 제공된 context만 근거로 답변하라.
# context에 없으면 추측하지 말고 "제공된 문서에서 확인되지 않습니다."라고 답하라.
# 답변은 한국어로, 핵심부터 간결하게 작성하라.
# <context>
# {context}
# </context>

# 질문: {question}
# """
# )


# rag_chain = (
#     multi_retriever
#     | RunnableLambda(extract_metadata)
#     | {
#         "context": lambda x: x["context"],
#         "regulation": lambda x: x["regulation"],
#         "article": lambda x: x["article"],
#         "input": RunnablePassthrough()
#     }
#     | qa_prompt
#     | llm
# )

# rag_chain = (
#     {
#         "context": multi_retriever | RunnableLambda(format_docs),
#         "input": RunnablePassthrough(),
#     }
#     | qa_prompt
#     | llm
# )


rag_chain = (
    {
        "docs": multi_retriever,
        "input": RunnablePassthrough(),
    }
    | RunnableLambda(extract_metadata)
    | qa_prompt
    | llm
)


In [91]:
# query = "통신설비의 접지 설치 시 접지의 깊이는 얼마인가?"
# query = "통신선과 전원선의 이격거리는?"
# query = "통신설비의 접지저항은 얼마로 해야하나요?"
query = "구내통신실의 면적기준에 대해 설명해주세요?"

result = rag_chain.invoke(query)

print("result :",result.content)

doc

 제19조(구내통신실의 면적확보) 「전기통신사업법」
doc

 [제목개정 2017. 4. 25.]



제19조(구내통신실의 면적확보) 「전기통신사업법」 제69조제2항에 따른 전기통신회선설비와의 접속을 위한 면적기준은 다음 각 호와 같다. <개정 2011. 1. 4., 2017. 4. 25., 2022. 12. 6.>

1. 업무용건축물에는 국선ㆍ국선단자함 또는 국선배선반과 초고속통신망장비, 이동통신망장비 등 각종 구내통신선로설비 및 구내용 이동통신설비를 설치하기 위한 공간으로서 다음 각 목의 구분에 따라 집중구내통신실과 층구내통신실을 확보하여야 한다.

가. 집중구내통신실: 별표 2에 따른 면적확보 기준을 충족할 것

나. 층구내통신실: 각 층별로 별표 2에 따른 면적확보 기준을 충족할 것

2. 주거용건축물 중 공동주택 및 준주택오피스텔에는 별표 3에 따른 면적확보 기준을 충족하는 집중구내통신실을 확보해야 한다.

3. 하나의 건축물에 업무용건축물과 주거용건축물 중 공동주택 또는 준주택오피스텔이 복합된 건축물에는 각각 별표 2 및 별표 3에 따른 면적확보 기준을 충족하는 집중구내통신실을 용도별로 각각 분리된 공간에 확보해야 하며, 업무용건축물에 해당하는 부분에는 별표 2에 따른 면적확보 기준을 충족하는 층구내통신실을 확보해야 한다. 다만, 업무용건축물에 해당하는 부분의 연면적이 500제곱미터 미만인 건축물로서 다음 각 목의 요건을 모두 충족하는 경우에는 집중구내통신실을 용도별로 분리하지 않고 통합된 공간에 확보할 수 있다.

가. 집중구내통신실의 면적이 별표 2와 별표 3에 따른 면적확보 기준을 합산한 면적 이상일 것
doc

 제2항에 따른 전기통신회선설비와의 접속을 위한 면적기준은 다음 각 호와 같다. <개정 2011. 1. 4., 2017. 4. 25., 2022. 12. 6.>

1. 업무용건축물에는 국선ㆍ국선단자함 또는 국선배선반과 초고속통신망장비, 이동통신망장비 등 각종 구내통신선로설비 및 구내용 이동통신설비를 설치하기 위한 공간으로서 